# Production RAG with Caching & Observability


- **Redis Caching**: Intelligent response caching built into RAG endpoints
- **LangFuse Observability**: End-to-end tracing and analytics
- **Performance Optimization**: Sub-second responses for cached queries
- **Production Monitoring**: Real-time metrics and debugging

---

## System Architecture

```
┌─────────────────┐     ┌─────────────────┐     ┌─────────────────┐
│   User Query    │────▶│  /api/v1/ask    │────▶│  Redis Cache    │
└─────────────────┘     └─────────────────┘     └─────────────────┘
                                │                         │
                                │                    Cache Hit?
                                │                         │
                         ┌──────┴──────┐        ┌────────┴────────┐
                         │             │        │                 │
                      Hit ▼          Miss ▼     ▼ Return Cached   │
                 ┌─────────────┐  ┌─────────────┐   (<100ms)      │
                 │Return Cache │  │   Search    │                 │
                 │  Response   │  │     +       │                 │
                 └─────────────┘  │    LLM      │                 │
                         │        │     +       │                 │
                         │        │  Store      │                 │
                         │        │  Cache      │                 │
                         │        └─────────────┘                 │
                         │                 │                      │
                         └─────────────────┼──────────────────────┘
                                           │
                                           ▼
                                  ┌─────────────────┐
                                  │    LangFuse     │
                                  │    (Tracing)    │
                                  └─────────────────┘
```

---


In [ ]:
# Check API Endpoints
print("API STRUCTURE")
print("=" * 20)

try:
    response = requests.get("http://localhost:8000/openapi.json")
    if response.status_code == 200:
        openapi_data = response.json()
        endpoints = list(openapi_data['paths'].keys())
        
        print(f"Total endpoints: {len(endpoints)}")
        print("\nAvailable endpoints:")
        for endpoint in sorted(endpoints):
            print(f"  • {endpoint}")
      
    else:
        print(f"Could not fetch API info: {response.status_code}")
except Exception as e:
    print(f"Error: {e}")

In [ ]:
# Check Cache Status
print("CACHE CONFIGURATION")
print("=" * 40)

try:
    # Get health status 
    response = requests.get("http://localhost:8000/api/v1/health")
    if response.status_code == 200:
        health_data = response.json()
        print(f"API Status: {health_data.get('status', 'unknown')}")
        print(f"Cache Integration: Built into RAG endpoints")
        print(f"Cache Type: Redis")
        print(f"Cache Strategy: Exact parameter matching")
        print(f"TTL: Configurable (default 24 hours)")
        
        print(f"\n✓ Cache system is integrated and ready")
    else:
        print("Could not fetch API status")
except Exception as e:
    print(f"Error checking cache: {e}")

print(f"\nℹ️ Cache Testing Strategy:")
print(f"  1. First query: Full RAG pipeline (cache miss)")
print(f"  2. Identical query: Cached response (cache hit)")  
print(f"  3. Different query: Full RAG pipeline (cache miss)")

## 1. API Structure Overview

In [ ]:
# First Query - Should NOT use cache
print("FIRST QUERY TEST (NO CACHE - BASELINE)")
print("=" * 50)

test_query = "What are the latest advances in transformer models for NLP?"
print(f"Query: {test_query}")
print(f"\nExpected: Full RAG pipeline execution (15-20 seconds)")
print("-" * 50)

start_time = time.time()

try:
    request_data = {
        "query": test_query,
        "top_k": 3,
        "use_hybrid": True,
        "model": "llama3.2:1b"
    }
    
    print("\nSending request...")
    response = requests.post(
        "http://localhost:8000/api/v1/ask",
        json=request_data,
        timeout=60
    )
    
    first_query_time = time.time() - start_time
    
    if response.status_code == 200:
        data = response.json()
        
        print(f"\n✓ Success!")
        print(f"Response Time: {first_query_time:.2f} seconds")
        
        print(f"\nAnswer Preview:")
        print("-" * 50)
        answer_preview = data['answer'][:400] if len(data['answer']) > 400 else data['answer']
        print(answer_preview + ("..." if len(data['answer']) > 400 else ""))
        print("-" * 50)
        
        print(f"\nMetadata:")
        print(f"  • Sources: {len(data.get('sources', []))} papers")
        print(f"  • Chunks used: {data.get('chunks_used', 0)}")
        print(f"  • Search mode: {data.get('search_mode', 'hybrid')}")
        
        # Store for comparison
        first_answer = data['answer']
        first_response_data = data
        
    else:
        print(f"\n✗ Request failed: {response.status_code}")
        print(f"Response: {response.text[:200]}")
        first_query_time = None
        
except Exception as e:
    print(f"\n✗ Error: {e}")
    first_query_time = None

if first_query_time:
    print(f"\n📊 Baseline established: {first_query_time:.2f} seconds")

In [1]:
# Second Query - Should USE cache
print("SECOND QUERY TEST (WITH CACHE - OPTIMIZED)")
print("=" * 50)

# Same query as before
print(f"Query: {test_query}")
print(f"\nExpected: Cache hit (sub-second response)")
print("-" * 50)

# Small delay to ensure cache is written
time.sleep(0.5)

start_time = time.time()

try:
    request_data = {
        "query": test_query,
        "top_k": 3,
        "use_hybrid": True,
        "model": "llama3.2:1b"
    }
    
    print("\nSending identical request...")
    response = requests.post(
        "http://localhost:8000/api/v1/ask",
        json=request_data,
        timeout=60
    )
    
    second_query_time = time.time() - start_time
    
    if response.status_code == 200:
        data = response.json()
        
        print(f"\n✓ Success!")
        print(f"Response Time: {second_query_time:.3f} seconds ({second_query_time*1000:.0f}ms)")
        
        print(f"\nAnswer Preview:")
        print("-" * 50)
        answer_preview = data['answer'][:400] if len(data['answer']) > 400 else data['answer']
        print(answer_preview + ("..." if len(data['answer']) > 400 else ""))
        print("-" * 50)
        
        # Store for comparison
        second_answer = data['answer']
        
        # Performance comparison
        if first_query_time and second_query_time:
            speedup = first_query_time / second_query_time
            time_saved = first_query_time - second_query_time
            
            print(f"\nPERFORMANCE COMPARISON")
            print("=" * 50)
            print(f"First Query (no cache): {first_query_time:.2f} seconds")
            print(f"Second Query (cached): {second_query_time:.3f} seconds")
            print(f"\nSpeed Improvement: {speedup:.0f}x faster")
            print(f"⏱Time Saved: {time_saved:.2f} seconds")
            
            # Verify answers are identical
            if first_answer == second_answer:
                print(f"\n✓ Answers are identical (cache working correctly)")
            else:
                print(f"\n⚠ Answers differ (cache may not be active)")
            
            if speedup > 50:
                print(f"\nAchieved {speedup:.0f}x performance improvement!")
                print(f"   This demonstrates production-grade caching!")
        
    else:
        print(f"\n✗ Request failed: {response.status_code}")
        second_query_time = None
        
except Exception as e:
    print(f"\n✗ Error: {e}")
    second_query_time = None

SECOND QUERY TEST (WITH CACHE - OPTIMIZED)


NameError: name 'test_query' is not defined